In [1]:
import os
import numpy as np
from tqdm import tqdm

In [2]:
DATA_DIR = "./data/raw"
OUT_DIR = "./data/processed"
os.makedirs(OUT_DIR, exist_ok=True)

K0_PHASE_C = 4.67
K0_PHASE_A = 4.93
K0_EPS = 1e-6
DOWNSAMPLE = 20

In [3]:
def parse_nso(line: str):
    """
    Parse a global-observable line starting with '# Nso'.

    Example input line:
        # Nso 16344 104277 253668 110490 100028 1.163670 3740

    According to the CDT data format, we:
      - take all numeric values after 'Nso'
      - drop the last two values
      - keep the very last value
    This selects the subset of global observables used in the paper.

    Returns:
        List[float]: selected global observables
    """
    # split line into tokens
    parts = line.split()

    # find the position of the 'Nso' keyword
    idx = parts.index("Nso")

    # take everything after 'Nso'
    after = parts[idx + 1:]

    # keep all but the last two entries, and also keep the final entry
    values = after[:-2] + after[-1:]

    # convert all values to float
    return [float(v) for v in values]


def parse_vto(line: str):
    """
    Parse a local-observable line starting with 'Vto'.

    Example input line:
        Vto t x1 x2 x3 x4 x5 x6

    The first two entries ('Vto' and time index t) are discarded.
    Only the six local geometric observables are returned.

    Returns:
        List[float]: local observables for a single time slice
    """
    # split line into tokens
    parts = line.split()

    # skip 'Vto' and time index, keep x1..x6
    return [float(v) for v in parts[2:]]


def parse_k0_from_filename(filename: str) -> float:
    """
    Extract the coupling constant κ₀ from the filename.

    Example filename:
        vto-4.67-0.6-T4-100k-torus-L.out

    The second '-' separated field encodes κ₀.

    Returns:
        float: κ₀ value
    """
    return float(filename.split("-")[1])


def flatten_sample(sample):
    """
    Convert a single CDT configuration into a flat feature vector.

    The feature vector consists of:
      - global observables (Nso)
      - local observables (Vto), ordered by discrete time slice

    This ordering enforces time-translation symmetry after
    cyclic time-shift augmentation.

    Returns:
        List[float]: 1D feature vector (length = 30)
    """
    features = []

    # add global observables
    features.extend(sample["Nso"])

    # add local observables in time order
    for vto_t in sample["Vto"]:
        features.extend(vto_t)

    return features


def label_from_k0(k0):
    """
    Assign a phase label based on κ₀.

    Deep inside phase C:
        label = 0
    Deep inside phase A:
        label = 1
    Intermediate κ₀ values:
        label = None (not used for training)

    A small tolerance is used to avoid floating-point issues.

    Returns:
        int or None
    """
    if abs(k0 - K0_PHASE_C) < K0_EPS:
        return 0
    if abs(k0 - K0_PHASE_A) < K0_EPS:
        return 1
    return None

In [4]:
files = sorted(
    f for f in os.listdir(DATA_DIR)
    if f.startswith("vto-") and f.endswith(".out")
)

In [5]:
files

['vto-4.67-0.6-T4-100k-torus-L.out',
 'vto-4.70-0.6-T4-100k-torus-L.out',
 'vto-4.72-0.6-T4-100k-torus-L.out',
 'vto-4.73-0.6-T4-100k-torus-L.out',
 'vto-4.74-0.6-T4-100k-torus-L.out',
 'vto-4.75-0.6-T4-100k-torus-L.out',
 'vto-4.76-0.6-T4-100k-torus-L.out',
 'vto-4.77-0.6-T4-100k-torus-L.out',
 'vto-4.78-0.6-T4-100k-torus-L.out',
 'vto-4.79-0.6-T4-100k-torus-L.out',
 'vto-4.81-0.6-T4-100k-torus-L.out',
 'vto-4.82-0.6-T4-100k-torus-LL.out',
 'vto-4.83-0.6-T4-100k-torus-LL.out',
 'vto-4.84-0.6-T4-100k-torus-L.out',
 'vto-4.85-0.6-T4-100k-torus-LL.out',
 'vto-4.86-0.6-T4-100k-torus-LL.out',
 'vto-4.87-0.6-T4-100k-torus-LL.out',
 'vto-4.88-0.6-T4-100k-torus-LL.out',
 'vto-4.90-0.6-T4-100k-torus-LL.out',
 'vto-4.93-0.6-T4-100k-torus-LL.out']

In [6]:
# Containers for the final dataset (filled after concatenation)
X_full = []
y_full = []
K0_full = []

# Separate buffers for each time-shift variant
# shift = 0, 1, 2, 3 correspond to cyclic time translations
X_shifts = [[], [], [], []]
y_shifts = [[], [], [], []]
K0_shifts = [[], [], [], []]


# Loop over all CDT output files (each file corresponds to a fixed κ₀)
for file_idx, filename in enumerate(tqdm(files, desc="Parsing files")):

    # Number of initial Monte Carlo configurations to discard
    # (thermalization cut; endpoints use a slightly smaller cut)
    SKIP_SAMPLES = 350_000 if file_idx in (0, len(files) - 1) else 400_000

    # Extract κ₀ value from filename and assign phase label (if deep A or C)
    file_k0 = parse_k0_from_filename(filename)
    file_label = label_from_k0(file_k0)

    file_path = os.path.join(DATA_DIR, filename)

    # Temporary storage for the currently parsed configuration
    current_sample = None
    sample_counter = 0

    # Read the file line by line
    with open(file_path, "r") as f:
        for line in f:
            line = line.strip()

            # Marker for the beginning of a new configuration
            if line.startswith("# ntime"):

                # If a previous configuration was fully read, process it
                if current_sample is not None:
                    sample_counter += 1

                    # Skip non-equilibrated configurations
                    if sample_counter > SKIP_SAMPLES:

                        # Apply cyclic time-shift augmentation (×4)
                        for shift in range(4):
                            perm = {
                                # Global observables are unchanged by time shifts
                                "Nso": current_sample["Nso"],

                                # Local observables are cyclically shifted in time
                                "Vto": (
                                    current_sample["Vto"][shift:]
                                    + current_sample["Vto"][:shift]
                                )
                            }

                            # Store features, κ₀ value, and label
                            X_shifts[shift].append(flatten_sample(perm))
                            K0_shifts[shift].append(file_k0)
                            y_shifts[shift].append(file_label)

                # Initialize a new configuration container
                current_sample = {
                    "Nso": None,   # global observables
                    "Vto": []      # list of local observables (one per time slice)
                }

            # Parse global observables
            elif line.startswith("# Nso"):
                current_sample["Nso"] = parse_nso(line)

            # Parse local observables for a single time slice
            elif line.startswith("Vto"):
                current_sample["Vto"].append(parse_vto(line))

    # Report number of equilibrated configurations processed in this file
    print(f"{filename}: parsed {sample_counter - SKIP_SAMPLES} samples")


# After all files are processed, concatenate time-shift buffers
# The final ordering is:
#   all shift-0 samples, then all shift-1, shift-2, shift-3 samples
X_full = np.concatenate(
    [np.asarray(X_shifts[s]) for s in range(4)],
    axis=0
)

K0_full = np.concatenate(
    [np.asarray(K0_shifts[s]) for s in range(4)],
    axis=0
)

y_full = np.concatenate(
    [np.asarray(y_shifts[s], dtype=object) for s in range(4)],
    axis=0
)


Parsing files:   5%|█▍                           | 1/20 [00:04<01:29,  4.70s/it]

vto-4.67-0.6-T4-100k-torus-L.out: parsed 94149 samples


Parsing files:  10%|██▉                          | 2/20 [00:09<01:21,  4.55s/it]

vto-4.70-0.6-T4-100k-torus-L.out: parsed 50654 samples


Parsing files:  15%|████▎                        | 3/20 [00:13<01:18,  4.59s/it]

...

vto-4.88-0.6-T4-100k-torus-LL.out: parsed 89612 samples


Parsing files:  95%|██████████████████████████▌ | 19/20 [01:33<00:05,  5.01s/it]

vto-4.90-0.6-T4-100k-torus-LL.out: parsed 91068 samples


Parsing files: 100%|████████████████████████████| 20/20 [01:40<00:00,  5.00s/it]

vto-4.93-0.6-T4-100k-torus-LL.out: parsed 141961 samples


In [7]:
X_full = np.asarray(X_full)
y_full = np.asarray(y_full, dtype=object)
K0_full = np.asarray(K0_full)

np.savez(
    os.path.join(OUT_DIR, "dataset_full.npz"),
    X=X_full,
    y=y_full,
    K0=K0_full
)

print("dataset_full.npz zapisany")
print("X_full shape:", X_full.shape)


dataset_full.npz zapisany
X_full shape: (6205696, 30)


In [8]:
mask_train = (
    (K0_full == K0_PHASE_C) |
    (K0_full == K0_PHASE_A)
) & (y_full != None)

X_train = X_full[mask_train]
y_train = y_full[mask_train].astype(int)
K0_train = K0_full[mask_train]

# downsampling
idx = np.arange(len(X_train)) % DOWNSAMPLE == 0

X_train = X_train[idx]
y_train = y_train[idx]
K0_train = K0_train[idx]

np.savez(
    os.path.join(OUT_DIR, "dataset_train_shortened.npz"),
    X=X_train,
    y=y_train,
    K0=K0_train
)

print("dataset_train_shortened.npz zapisany")
print("X_train shape:", X_train.shape)


dataset_train_shortened.npz zapisany
X_train shape: (47222, 30)
